# Trabajo en clase — Q-Learning con FrozenLake

En CheeseWorld construimos el algoritmo desde cero. Ahora utilizaremos el mismo procedimiento en un ambiente estándar de **Gymnasium**.

## Objetivo

Durante la clase debes relacionar cada parte del código con los conceptos:

- estado \(s\);
- acción \(a\);
- recompensa \(r\);
- Q-table;
- exploración y explotación;
- TD target;
- TD error;
- política greedy.

Este notebook tiene **espacios para discutir y escribir conclusiones durante la clase**.


## 1. Imports y funciones de Q-Learning


In [ ]:
import numpy as np
import gymnasium as gym
import random
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output, display

from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt


def play_episode(env, Q=None, random_policy=False, max_steps=100, seed=None):
    """Ejecuta un episodio y devuelve sus frames y recompensa total."""
    state, _ = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0

    for _ in range(max_steps):
        if random_policy:
            action = env.action_space.sample()
        else:
            q_values = Q[state]
            max_q = np.max(q_values)

            # Desempate aleatorio entre acciones con el mismo Q.
            best_actions = np.flatnonzero(q_values == max_q)
            action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())
        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=700):
    """Convierte una lista de frames RGB en una animación reproducible en Jupyter."""
    fig = plt.figure(figsize=(4, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))


def greedy_policy(Qtable, state):
    # Si hay empate entre varias acciones con el mismo Q, desempata al azar.
    max_q = np.max(Qtable[state])
    best_actions = np.flatnonzero(Qtable[state] == max_q)
    return int(np.random.choice(best_actions))


def epsilon_greedy_policy(Qtable, state, epsilon, env):
    if random.random() < epsilon:
        return env.action_space.sample()

    return greedy_policy(Qtable, state)


def train_q_learning(
    env,
    Qtable,
    n_episodes=5000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    max_steps=100,
    start_episode=0,
):
    rewards = []

    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        global_episode = start_episode + episode
        epsilon = min_epsilon + (
            max_epsilon - min_epsilon
        ) * np.exp(-decay_rate * global_episode)

        for _ in range(max_steps):
            action = epsilon_greedy_policy(
                Qtable, state, epsilon, env
            )

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            best_next_q = 0.0 if done else np.max(Qtable[next_state])

            td_target = reward + gamma * best_next_q
            td_error = td_target - Qtable[state, action]

            Qtable[state, action] += learning_rate * td_error

            state = next_state
            total_reward += reward

            if done:
                break

        rewards.append(total_reward)

    return Qtable, rewards


def evaluate_q_policy(env, Qtable, n_episodes=100, max_steps=100):
    episode_rewards = []

    for _ in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            action = greedy_policy(Qtable, state)

            next_state, reward, terminated, truncated, _ = env.step(action)

            state = next_state
            total_reward += reward

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)


def show_frame(env, title=''):
    frame = env.render()
    plt.figure(figsize=(4, 4))
    plt.imshow(frame)
    plt.axis('off')
    plt.title(title)
    display(plt.gcf())
    plt.close()

### 💬 Antes de ejecutar

En CheeseWorld teníamos explícitamente una clase `Environment` y una clase `QLearningAgent`.

**Pregunta:** en este notebook, ¿qué papel cumple Gymnasium y dónde quedó representado el agente?

**Respuesta:**

Gymnasium proporciona el ambiente FrozenLake: define los estados, las acciones, las recompensas y lo que sucede cuando el agente se mueve. Con env.reset() iniciamos un episodio y con env.step(action) ejecutamos una acción y observamos su resultado.

El agente está representado por la Q-table y las funciones que seleccionan acciones y actualizan sus valores. Aunque no existe una clase QLearningAgent, sus responsabilidades aparecen en epsilon_greedy_policy, greedy_policy y train_q_learning.

El ambiente responde a las acciones; el agente aprende a elegirlas mediante la experiencia.

## 2. Crear FrozenLake


In [ ]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="rgb_array"
)

state, info = env.reset()

print("Estado inicial:", state)
print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)

En FrozenLake las acciones son:

| Acción | Código |
|---|---:|
| Left | 0 |
| Down | 1 |
| Right | 2 |
| Up | 3 |

Primero trabajaremos con `is_slippery=False`, es decir, con transiciones determinísticas.


### 💬 Actividad 1 — La Q-table

Antes de crearla:

1. ¿Cuántas filas debe tener la Q-table?
2. ¿Cuántas columnas?
3. ¿Qué representa una celda \(Q[s,a]\)?

**Respuesta / discusión:**

1. La Q-table debe tener **16 filas**, porque el mapa de FrozenLake es de 4 × 4 y cada casilla representa un estado.
2. Debe tener **4 columnas**, una por cada acción: izquierda, abajo, derecha y arriba.
3. Cada celda Q[s,a] estima la recompensa acumulada y descontada que se puede obtener al realizar la acción a desde el estado s y continuar con las mejores acciones.

Por lo tanto, la tabla tiene forma **`(16, 4)`**. Las filas de los estados terminales también están incluidas, aunque no se toman nuevas decisiones desde ellos.


In [ ]:
state_space = env.observation_space.n
action_space = env.action_space.n

Q = initialize_q_table(state_space, action_space)

print("Q-table shape:", Q.shape)
Q

### 💬 Actividad 2 — Inicio del aprendizaje

Todos los valores son cero.

$$
Q(s,a)=0
$$

¿Esto significa que todas las acciones son malas, o que el agente todavía no sabe nada?

¿Qué ocurre si varias acciones tienen exactamente el mismo valor máximo?

**Notas:**

Que todos los valores Q sean cero no significa que todas las acciones sean malas. Significa que el agente comienza con estimaciones iniciales de cero y todavía no ha aprendido de sus experiencias.

Cuando varias acciones tienen el mismo valor máximo, el código selecciona aleatoriamente una de las acciones empatadas. Al principio, como todas tienen el mismo valor, cualquiera puede ser elegida.

A medida que el agente interactúa con el ambiente, actualiza la Q-table y empieza a distinguir cuáles acciones ofrecen mejores resultados esperados.

## 3. Ejecutar una transición


In [ ]:
state, _ = env.reset()

epsilon = 1.0
action = epsilon_greedy_policy(Q, state, epsilon, env)

next_state, reward, terminated, truncated, _ = env.step(action)

print("state      =", state)
print("action     =", action)
print("reward     =", reward)
print("next_state =", next_state)
print("done       =", terminated or truncated)

In [ ]:
nombres_acciones = {
    0: "izquierda",
    1: "abajo",
    2: "derecha",
    3: "arriba"
}

print("Experiencia (s, a, r, s'):")
print((state, action, reward, next_state))
print("Acción ejecutada:", nombres_acciones[int(action)])
print("¿Terminó el episodio?:", terminated or truncated)

### 💬 Actividad 3 — Identificar la experiencia

Escribe la experiencia anterior como:

$$
(s,a,r,s')
$$

**Experiencia:**

 (s, a, r, s')

¿De cuál de esos cuatro elementos **no disponíamos directamente** en Value Iteration cuando hablábamos de experiencia real?


**Interpretación de la experiencia:**

La experiencia contiene el estado inicial s, la acción ejecutada a, la recompensa observada r y el estado al que llegamos s'.

En Value Iteration no necesitábamos ejecutar una acción real para observar un estado siguiente concreto. Utilizábamos un modelo con los posibles estados siguientes, sus probabilidades y sus recompensas. En Q-Learning, el estado siguiente s' y la recompensa r se obtienen directamente al interactuar con el ambiente.

Por eso, la diferencia principal es que aquí aprendemos a partir de una transición observada, en lugar de calcular el resultado usando todo el modelo.

## 4. Del azar a una política aprendida

Vamos a observar **el mismo agente en tres momentos**. Primero no sabe nada y actúa al azar; luego veremos su política después de pocas experiencias; finalmente veremos la política después del entrenamiento completo.


In [ ]:
# Guardaremos tres momentos del aprendizaje

# Momento 1: sin entrenamiento
Q_initial = initialize_q_table(
    env.observation_space.n,
    env.action_space.n
)

# Momento 2: poco entrenamiento
EARLY_EPISODES = 50
Q_early = Q_initial.copy()
Q_early, rewards_early = train_q_learning(
    env,
    Q_early,
    n_episodes=EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
)

# Momento 3: continuar hasta 10 000 episodios
TOTAL_EPISODES = 10000
Q_trained = Q_early.copy()
Q_trained, rewards_final = train_q_learning(
    env,
    Q_trained,
    n_episodes=TOTAL_EPISODES - EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    start_episode=EARLY_EPISODES,
)

Q = Q_trained
rewards = rewards_early + rewards_final

print('Snapshots guardados:')
print('Q_initial : 0 episodios')
print(f'Q_early   : {EARLY_EPISODES} episodios')
print(f'Q_trained : {TOTAL_EPISODES} episodios')


### Momento 1 — Sin entrenamiento: random walk
Todavía no usamos la Q-table para decidir. Cada acción se selecciona aleatoriamente. Observa cómo interactúa el agente con el mundo.

In [ ]:
frames_random, reward_random = play_episode(
    env,
    Q_initial,
    random_policy=True,
    seed=7
)

print(f"Recompensa total: {reward_random}")
frames_to_video(frames_random, interval=700)


### Momento 2 — Después de pocas iteraciones

Ahora el agente usa de forma **greedy** lo que ha aprendido en `Q_early`. Todavía conoce poco del ambiente, así que su comportamiento puede ser incompleto o equivocarse.


In [ ]:
frames_early, reward_early = play_episode(
    env,
    Q_early,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_early}")
frames_to_video(frames_early, interval=700)


### Momento 3 — Agente entrenado

Finalmente usamos `Q_trained`. Ya no exploramos: en cada estado el agente selecciona una de las acciones con mayor valor $Q(s,a)$.


In [ ]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_trained}")
frames_to_video(frames_trained, interval=700)


### 💬 Actividad — ¿Qué cambió?

Compara las tres ejecuciones. El ambiente, los estados y las acciones son los mismos. **¿Qué cambió internamente en el agente para que su comportamiento mejore?**

Observa `Q_initial`, `Q_early` y `Q_trained` y relaciona sus valores con las acciones que viste ejecutar.


**Respuesta:**

Lo que cambió internamente fueron los valores de la Q-table. El ambiente conserva los mismos estados, acciones y reglas, pero el agente ajusta sus estimaciones a partir de las experiencias que obtiene.

En Q_initial, todos los valores son cero porque todavía no existe aprendizaje. En Q_early, después de 50 episodios, algunas acciones pueden tener valores diferentes si el agente ha obtenido información útil durante la exploración. En Q_trained, después de 10 000 episodios, ha tenido más oportunidades de aprender qué acciones permiten llegar a la meta.

La política greedy utiliza esos valores para seleccionar una acción con el mayor Q en cada estado. El entrenamiento busca mejorar las decisiones, aunque una ejecución aislada no basta para medir el desempeño: por eso también evaluamos la política durante varios episodios.

In [ ]:
s = 0
nombres_acciones = ["Left", "Down", "Right", "Up"]

valores = Q_trained[s]
maximo = np.max(valores)
mejores = np.flatnonzero(valores == maximo)

print("Estado seleccionado:", s)
print("\nValores Q:")

for nombre, valor in zip(nombres_acciones, valores):
    print(f"- {nombre}: {valor:.6f}")

acciones_mejores = ", ".join(
    nombres_acciones[int(a)] for a in mejores
)

print("\nAcción o acciones con mayor Q:", acciones_mejores)

print("\nInterpretación:")
if len(mejores) == 1:
    print(
        f"La política greedy seleccionaría {acciones_mejores}, "
        "porque tiene el mayor valor Q en este estado. "
        "Según lo aprendido, esta acción ofrece la mayor "
        "recompensa acumulada y descontada esperada."
    )
else:
    print(
        f"La política greedy elegiría aleatoriamente entre "
        f"{acciones_mejores}, porque sus valores Q están "
        "empatados en el máximo."
    )

### 💬 Actividad 4 — Leer una fila de Q

Selecciona un estado $s$ y observa:

$$
Q(s,0), Q(s,1), Q(s,2), Q(s,3)
$$

**Estado seleccionado:** 0

**Valores Q:**

- Left: 0.735092
- Down: 0.773781
- Right: 0.773781
- Up: 0.735092

¿Cuál acción seleccionaría:

$$
\arg\max_a Q(s,a)
$$
= Acción o acciones con mayor Q: Down, Right


**Interpretación:**


Desde el estado 0, Down y Right tienen el mayor valor Q mostrado: 0.773781. Por eso, ambas representan las mejores alternativas según estos valores. Si están exactamente empatadas en la Q-table, la política greedy selecciona aleatoriamente una de ellas. Left y Up tienen un valor menor porque, desde la esquina inicial, mantienen al agente en la misma casilla y retrasan la llegada a la recompensa.

## 5. Evaluar la política aprendida


In [ ]:
mean_reward, std_reward = evaluate_q_policy(
    env,
    Q_trained,
    n_episodes=100
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")


### 💬 Actividad 5 — Exploration vs. exploitation

Durante entrenamiento usamos $\epsilon$-greedy.

Durante evaluación usamos:

$$
a=\arg\max_aQ(s,a)
$$

¿Por qué **no exploramos** durante la evaluación?

**Conclusión:**

Durante la evaluación no exploramos porque queremos medir el desempeño de la política que el agente ya aprendió. Por eso seleccionamos las acciones con mayor valor Q, sin introducir acciones aleatorias para descubrir nuevas opciones.

Si mantuviéramos la exploración, los resultados incluirían decisiones que el agente toma para aprender y no necesariamente las que considera mejores. Además, durante la evaluación no actualizamos la Q-table.

El código todavía puede elegir aleatoriamente entre acciones empatadas en el máximo, pero ese desempate no corresponde a la exploración de epsilon-greedy.


## 6. Experimento: FrozenLake estocástico


In [ ]:
slippery_env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True,
    render_mode="rgb_array"
)

Q_slippery = initialize_q_table(
    slippery_env.observation_space.n,
    slippery_env.action_space.n
)

Q_slippery, rewards_slippery = train_q_learning(
    slippery_env,
    Q_slippery,
    n_episodes=20000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.0005,
    max_steps=100
)

mean_reward, std_reward = evaluate_q_policy(
    slippery_env,
    Q_slippery,
    n_episodes=500
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")

### 💬 Actividad 6 — Determinístico vs. estocástico

Compara:

- `is_slippery=False`
- `is_slippery=True`

¿Qué cambia en el **ambiente**?

¿Qué cambia en la **ecuación de Q-Learning**?

**Discusión:**

- **Ambiente:** con is_slippery=False, el movimiento sigue la dirección elegida, salvo cuando un borde impide avanzar. Con is_slippery=True, el agente puede deslizarse y terminar en una dirección diferente. Por eso, una misma acción desde un mismo estado puede producir distintos estados siguientes.

- **Algoritmo:** la ecuación de Q-Learning no cambia. Seguimos actualizando Q(s,a) con la recompensa observada y el mayor valor Q del siguiente estado. Lo que cambia son las experiencias que produce el ambiente y, por lo tanto, los valores que aprende el agente.

En el ambiente resbaloso, elegir una buena acción no garantiza llegar a la meta en cada episodio. La recompensa media permite observar la proporción de episodios exitosos, porque en este FrozenLake la meta entrega 1 y los demás resultados entregan 0. La desviación estándar muestra cuánto varían esos resultados.

Los dos entrenamientos utilizan diferentes cantidades de episodios y tasas de reducción de epsilon, por lo que sus resultados no permiten atribuir todas las diferencias únicamente al deslizamiento.

# Cierre de clase

Completa antes de terminar:

**1. ¿Qué almacena $Q(s,a)$?**

>

**2. ¿De dónde sale $\max_{a'}Q(s',a')$?**

>

**3. ¿Por qué necesitamos $\epsilon$-greedy?**

>

**4. ¿Por qué Q-Learning es model-free?**

>


# Respuestas

**1. ¿Qué almacena Q(s,a)?**

Almacena una estimación de la recompensa acumulada y descontada que se puede obtener al ejecutar la acción a desde el estado s y continuar con las mejores acciones. Su valor se ajusta mediante las experiencias del agente.

**2. ¿De dónde sale el máximo de Q(s',a')?**

Sale de revisar la fila de la Q-table correspondiente al siguiente estado s' y seleccionar el mayor valor entre sus acciones. En el código se calcula con np.max(Qtable[next_state]). Representa la mejor recompensa futura estimada desde ese estado. Cuando la transición termina el episodio, este notebook utiliza un valor futuro de cero.

**3. ¿Por qué necesitamos epsilon-greedy?**

Porque necesitamos combinar exploración y explotación. Explorar permite probar acciones y descubrir rutas que el agente todavía no conoce; explotar permite aprovechar las acciones que tienen los mejores valores aprendidos. Sin exploración, el agente podría quedarse con decisiones poco convenientes por no haber probado otras alternativas. En este notebook, epsilon disminuye durante el entrenamiento, aunque conserva un mínimo de 0.05.

**4. ¿Por qué Q-Learning es model-free?**

Porque el agente no necesita conocer previamente las probabilidades de transición ni una función completa de recompensas para actualizar la Q-table. Aprende usando las experiencias que obtiene al ejecutar acciones: el estado, la acción, la recompensa y el siguiente estado. Aunque Gymnasium implementa las reglas del ambiente, el algoritmo de aprendizaje no consulta ese modelo para calcular sus actualizaciones.

## Lo que aprendí, Isabella Cadavid Posada

Con este notebook comprendí cómo aplicar Q-Learning a un ambiente de Gymnasium y cómo relacionar la Q-table con las decisiones del agente. Aprendí que explorar permite descubrir opciones y que aprovechar los valores aprendidos ayuda a elegir mejores acciones. También entendí que el deslizamiento introduce incertidumbre en los movimientos, pero no cambia la ecuación del algoritmo. Por eso, es necesario evaluar varios episodios para conocer el desempeño de la política y no sacar conclusiones únicamente de un recorrido.